# Hiris & Blake (1992), Fig. 1: the dynamic motion aftereffect in units of real motion

For each session: the proportion of trials on which the **adapted (variable) display** was judged
more coherently downward than the **40 % reference**, against the adapted display's signed signal
(+ downward, the aftereffect's direction; - upward). A cumulative normal is fitted by maximum
likelihood (the paper: "probit analysis"). The point of subjective equality (PSE) is the signal
that looks as coherent as the reference.

**Aftereffect strength = PSE(no adaptation) - PSE(adapted to upward motion)**, in % signal.
The paper's adapted observers matched RDVN (0 %) with 40 % real downward signal (p. 9026).

Set `ANALYSIS_RUN` to one or more run folders (separated by the OS path separator), or leave it
unset to read every run under `../data`. Simulated sessions are rehearsals: their numbers come
from a planted observer, not from anyone.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from hirisblake1992 import analysis

env = os.environ.get("ANALYSIS_RUN")
if env:
    runs = [Path(p) for p in env.split(os.pathsep) if p]
else:
    runs = sorted({p.parent for p in Path("../data").rglob("*_trials.csv")})
trials = analysis.load_runs(runs)
print(f"{len(runs)} run(s), {len(trials)} trials, modes: {sorted(trials['adapt_mode'].unique())}")

In [ ]:
table = analysis.psychometric_table(trials)
table

In [ ]:
fits = {}
for mode in sorted(table["adapt_mode"].unique()):
    fits[mode] = analysis.fit_mode(trials, mode)
    f = fits[mode]
    print(f"{mode:15s} PSE {f.pse:6.1f} %   sigma {f.sigma:6.1f} %   n {f.n}")

In [ ]:
labels = {"unidirectional": "adapted to upward motion", "random": "no adaptation (RDVN)"}
colors = {"unidirectional": "#7f1d1d", "random": "#5b6169"}
fig, ax = plt.subplots(figsize=(6, 4))
x = np.linspace(table["test_signal_pct"].min() - 5, table["test_signal_pct"].max() + 5, 300)
for mode, f in fits.items():
    t = table[table["adapt_mode"] == mode]
    ax.scatter(t["test_signal_pct"], t["p_test"], s=12 + 2 * t["n"], color=colors.get(mode),
               label=f"{labels.get(mode, mode)} (PSE {f.pse:.0f} %)")
    ax.plot(x, f.p(x), color=colors.get(mode))
ax.axhline(0.5, color="#dde0e4", lw=0.8, zorder=0)
ax.set_xlabel("signal in the adapted display (%; - upward, + downward)")
ax.set_ylabel("p(adapted display more coherently downward)")
ax.set_ylim(-0.02, 1.02)
ax.legend(frameon=False, fontsize=8)
plt.show()

In [ ]:
strength = analysis.aftereffect_strength(trials, n_boot=200)
for mode in ("unidirectional", "random"):
    if mode in strength:
        lo, hi = strength[mode]["pse_ci95"]
        print(f"{mode:15s} PSE {strength[mode]['pse']:6.1f} %  95% CI [{lo:.1f}, {hi:.1f}]")
if "aftereffect_pct" in strength:
    lo, hi = strength["aftereffect_ci95"]
    print(f"aftereffect: {strength['aftereffect_pct']:.1f} % signal, 95% CI [{lo:.1f}, {hi:.1f}]")
else:
    print("aftereffect needs both an adapted and a control session")

**Reading it.** A PSE near 40 % without adaptation shows the two displays are judged alike when
physically alike. A lower PSE after adaptation means the aftereffect added apparent downward
motion to the adapted display; the difference is that aftereffect in % of real signal. Comparable
slopes (sigma) in the two conditions are what the paper reported (Fig. 1 legend).

**What it cannot establish.** A shifted PSE with a dynamic test does not by itself show *where* the
adaptation happened, and it does not measure the conventional static aftereffect (the paper's
point is that the two differ). Display luminance (the paper's 15 cd/m2 white) and dot rendering
are not calibrated by this software.